# 🚗 Accidentologie routière en France (2009–2023)
## Nettoyage, exploration & préparation du dashboard

**Auteur :** Diogo Almeida  
**Date :** Septembre 2026  
**Source :** [data.gouv.fr — Accidents corporels](https://www.data.gouv.fr/fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere/)

---

### Objectifs de ce notebook

1. **Charger et consolider** les 4 fichiers annuels (caractéristiques, lieux, véhicules, usagers) sur 15 ans
2. **Nettoyer** les données : valeurs manquantes, recodage, types
3. **Explorer** les facteurs de risque : temporels, géographiques, météo, profils usagers
4. **Requêter en SQL** les KPIs clés
5. **Exporter** un fichier consolidé prêt pour Looker Studio

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3
import warnings
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

# Chemins relatifs — jamais de chemins absolus
DATA_RAW = Path("..") / "data" / "raw"
DATA_PROCESSED = Path("..") / "data" / "processed"
ASSETS = Path("..") / "assets"

# Créer les répertoires s'ils n'existent pas
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ASSETS.mkdir(parents=True, exist_ok=True)

# Années à charger
YEARS = list(range(2009, 2024))

print(f"📁 Données brutes dans : {DATA_RAW.resolve()}")
print(f"📅 Période : {YEARS[0]}–{YEARS[-1]} ({len(YEARS)} ans)")

## 2. Chargement des données brutes

Les données sont réparties en **4 fichiers par an** :
- `caracteristiques` : date, heure, luminosité, météo, département, commune
- `lieux` : catégorie de route, profil, tracé, état de surface
- `vehicules` : catégorie de véhicule, manœuvre, obstacle
- `usagers` : gravité, sexe, année de naissance, catégorie (conducteur/passager/piéton)

La clé de jointure est `Num_Acc` (identifiant unique de l'accident).

In [ ]:
def load_file(file_type: str, year: int) -> pd.DataFrame | None:
    """
    Charge un fichier CSV pour un type et une année donnés.
    Gère les variations d'encodage et de séparateur entre les années.
    """
    filepath = DATA_RAW / f"{file_type}-{year}.csv"
    
    if not filepath.exists():
        return None
    
    # Essayer plusieurs encodages (les fichiers varient selon les années)
    for encoding in ["utf-8", "latin-1", "cp1252"]:
        for sep in [";", ","]:
            try:
                df = pd.read_csv(
                    filepath,
                    sep=sep,
                    encoding=encoding,
                    low_memory=False,
                )
                if len(df.columns) > 1:  # Vérifier que le séparateur est bon
                    df["annee_fichier"] = year
                    return df
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
    
    print(f"  ⚠️  Impossible de lire {filepath.name}")
    return None


def load_all_years(file_type: str) -> pd.DataFrame:
    """Charge et concatène un type de fichier sur toutes les années."""
    frames = []
    for year in tqdm(YEARS, desc=f"  {file_type}"):
        df = load_file(file_type, year)
        if df is not None:
            frames.append(df)
    
    if not frames:
        raise FileNotFoundError(
            f"Aucun fichier '{file_type}' trouvé dans {DATA_RAW}. "
            f"Lance d'abord : python data/download_data.py"
        )
    
    combined = pd.concat(frames, ignore_index=True)
    print(f"  → {len(combined):,} lignes chargées sur {len(frames)} années\n")
    return combined


print("📥 Chargement des fichiers...\n")

df_carac = load_all_years("caracteristiques")
df_lieux = load_all_years("lieux")
df_vehicules = load_all_years("vehicules")
df_usagers = load_all_years("usagers")

### 2.1 Aperçu des données brutes

In [ ]:
print("=== CARACTÉRISTIQUES ===")
print(f"Shape : {df_carac.shape}")
print(f"Colonnes : {list(df_carac.columns)}")
display(df_carac.head(3))

print("\n=== LIEUX ===")
print(f"Shape : {df_lieux.shape}")
print(f"Colonnes : {list(df_lieux.columns)}")
display(df_lieux.head(3))

print("\n=== VÉHICULES ===")
print(f"Shape : {df_vehicules.shape}")
print(f"Colonnes : {list(df_vehicules.columns)}")
display(df_vehicules.head(3))

print("\n=== USAGERS ===")
print(f"Shape : {df_usagers.shape}")
print(f"Colonnes : {list(df_usagers.columns)}")
display(df_usagers.head(3))

## 3. Nettoyage & harmonisation

### Problèmes connus sur ce dataset :
- **Noms de colonnes** qui changent entre les années (ex: `an` vs `AN`, `dep` vs `Dep`)
- **Encodage des variables catégorielles** : codes numériques sans labels
- **Valeurs manquantes** : `-1`, `0`, ou vide selon les colonnes
- **Format de l'heure** : `hrmn` est un entier (ex: `1430` = 14h30)
- **Géolocalisation** : `lat`/`long` parfois en degrés × 100000

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise les noms de colonnes en minuscules et sans espaces."""
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

# Appliquer à tous les DataFrames
df_carac = normalize_columns(df_carac)
df_lieux = normalize_columns(df_lieux)
df_vehicules = normalize_columns(df_vehicules)
df_usagers = normalize_columns(df_usagers)

print("✓ Colonnes normalisées")
print(f"  Caractéristiques : {list(df_carac.columns)}")

### 3.1 Nettoyage des caractéristiques

In [ ]:
# --- Année ---
# Harmoniser la colonne 'an' (parfois sur 2 chiffres, parfois 4)
if "an" in df_carac.columns:
    df_carac["an"] = df_carac["an"].apply(
        lambda x: x + 2000 if x < 100 else x
    )

# --- Heure ---
# Extraire heure et minute depuis 'hrmn' (format: 1430 = 14h30)
if "hrmn" in df_carac.columns:
    df_carac["hrmn"] = pd.to_numeric(df_carac["hrmn"], errors="coerce")
    df_carac["hrmn_heure"] = (df_carac["hrmn"] // 100).astype("Int64")
    df_carac["hrmn_minute"] = (df_carac["hrmn"] % 100).astype("Int64")
elif "hr" in df_carac.columns:
    df_carac["hrmn_heure"] = pd.to_numeric(df_carac["hr"], errors="coerce").astype("Int64")
    df_carac["hrmn_minute"] = pd.to_numeric(
        df_carac.get("mn", pd.Series(dtype="Int64")), errors="coerce"
    ).astype("Int64")

# Filtrer les heures invalides
df_carac.loc[~df_carac["hrmn_heure"].between(0, 23), "hrmn_heure"] = pd.NA

# --- Date ---
# Construire une colonne date exploitable
if all(col in df_carac.columns for col in ["an", "mois", "jour"]):
    df_carac["date"] = pd.to_datetime(
        df_carac[["an", "mois", "jour"]].rename(
            columns={"an": "year", "mois": "month", "jour": "day"}
        ),
        errors="coerce",
    )
    df_carac["jour_semaine"] = df_carac["date"].dt.day_name()

# --- Département ---
if "dep" in df_carac.columns:
    df_carac["dep"] = df_carac["dep"].astype(str).str.strip().str.zfill(2)
    # Gérer les DOM-TOM (971, 972, etc.)
    mask_metro = df_carac["dep"].str.match(r"^\d{3}$") & ~df_carac["dep"].str.startswith("97")
    df_carac.loc[mask_metro, "dep"] = df_carac.loc[mask_metro, "dep"].str[:2]
    

# --- Géolocalisation ---
for col in ["lat", "long"]:
    if col in df_carac.columns:
        df_carac[col] = pd.to_numeric(df_carac[col], errors="coerce")
        # Convertir si les coordonnées sont en format ×100000
        mask = df_carac[col].abs() > 1000
        df_carac.loc[mask, col] = df_carac.loc[mask, col] / 100_000

print("✓ Caractéristiques nettoyées")
print(f"  Période : {df_carac['an'].min()} – {df_carac['an'].max()}")
print(f"  Accidents : {df_carac['num_acc'].nunique():,}")

### 3.2 Nettoyage des usagers

In [ ]:
# --- Gravité (grav) ---
# 1 = Indemne, 2 = Tué, 3 = Hospitalisé, 4 = Blessé léger
if "grav" in df_usagers.columns:
    df_usagers["grav"] = pd.to_numeric(df_usagers["grav"], errors="coerce")

    # Labels lisibles
    GRAVITE_LABELS = {1: "Indemne", 2: "Tué", 3: "Hospitalisé", 4: "Blessé léger"}
    df_usagers["gravite_label"] = df_usagers["grav"].map(GRAVITE_LABELS)

# --- Âge ---
# Calculer l'âge à partir de l'année de naissance
if "an_nais" in df_usagers.columns and "annee_fichier" in df_usagers.columns:
    df_usagers["an_nais"] = pd.to_numeric(df_usagers["an_nais"], errors="coerce")
    df_usagers["age"] = df_usagers["annee_fichier"] - df_usagers["an_nais"]
    # Filtrer les âges aberrants
    df_usagers.loc[~df_usagers["age"].between(0, 110), "age"] = pd.NA

# --- Sexe ---
if "sexe" in df_usagers.columns:
    df_usagers["sexe"] = pd.to_numeric(df_usagers["sexe"], errors="coerce")
    SEXE_LABELS = {1: "Homme", 2: "Femme"}
    df_usagers["sexe_label"] = df_usagers["sexe"].map(SEXE_LABELS)

# --- Catégorie d'usager ---
if "catu" in df_usagers.columns:
    CATU_LABELS = {1: "Conducteur", 2: "Passager", 3: "Piéton"}
    df_usagers["catu_label"] = df_usagers["catu"].map(CATU_LABELS)

# --- Tranches d'âge ---
if "age" in df_usagers.columns:
    bins = [0, 17, 24, 34, 44, 54, 64, 74, 120]
    labels = ["0-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"]
    df_usagers["tranche_age"] = pd.cut(
        df_usagers["age"], bins=bins, labels=labels, right=True
    )

print("✓ Usagers nettoyés")
print(f"  Usagers : {len(df_usagers):,}")
print(f"  Gravité :\n{df_usagers['gravite_label'].value_counts().to_string()}")

### 3.3 Recodage des variables catégorielles (lieux)

In [ ]:
# --- Catégorie de route ---
if "catr" in df_lieux.columns:
    CATR_LABELS = {
        1: "Autoroute", 2: "Route nationale", 3: "Route départementale",
        4: "Voie communale", 5: "Hors réseau public", 6: "Parc de stationnement",
        9: "Autre",
    }
    df_lieux["catr"] = pd.to_numeric(df_lieux["catr"], errors="coerce")
    df_lieux["route_label"] = df_lieux["catr"].map(CATR_LABELS)

# --- État de surface ---
if "surf" in df_lieux.columns:
    SURF_LABELS = {
        1: "Normale", 2: "Mouillée", 3: "Flaques", 4: "Inondée",
        5: "Enneigée", 6: "Boue", 7: "Verglacée", 8: "Corps gras", 9: "Autre",
    }
    df_lieux["surf"] = pd.to_numeric(df_lieux["surf"], errors="coerce")
    df_lieux["surface_label"] = df_lieux["surf"].map(SURF_LABELS)

print("✓ Lieux nettoyés")
print(f"  Lignes : {len(df_lieux):,}")

## 4. Consolidation — Jointure des 4 tables

On fusionne les 4 fichiers sur la clé `num_acc` pour obtenir une table unique exploitable.

> **Attention au grain :** la table finale est au grain *usager* (un usager = une ligne).  
> Pour les analyses par accident, il faudra dédupliquer sur `num_acc`.

In [ ]:
# Vérifier la clé de jointure
for name, df in [("carac", df_carac), ("lieux", df_lieux), ("vehicules", df_vehicules), ("usagers", df_usagers)]:
    if "num_acc" in df.columns:
        print(f"  {name:12s} : {df['num_acc'].nunique():>10,} accidents uniques / {len(df):>12,} lignes")
    else:
        print(f"  ⚠️  {name} : colonne 'num_acc' absente — colonnes : {list(df.columns[:5])}")

In [ ]:
# Jointure progressive
# 1. Caractéristiques + Lieux (1:1 sur num_acc)
df = df_carac.merge(df_lieux, on="num_acc", how="left", suffixes=("", "_lieux"))

# 2. + Véhicules (1:N — un accident peut impliquer plusieurs véhicules)
# On garde id_vehicule pour la jointure avec usagers
vehicule_cols = ["num_acc", "id_vehicule"] + [
    c for c in df_vehicules.columns
    if c not in ["num_acc", "id_vehicule", "annee_fichier"]
    and c not in df.columns
]
vehicule_cols_existing = [c for c in vehicule_cols if c in df_vehicules.columns]
df = df.merge(
    df_vehicules[vehicule_cols_existing],
    on="num_acc",
    how="left",
    suffixes=("", "_veh"),
)

# 3. + Usagers (jointure sur num_acc + id_vehicule si disponible)
usager_cols = [c for c in df_usagers.columns if c not in df.columns or c in ["num_acc", "id_vehicule"]]
join_keys = ["num_acc"]
if "id_vehicule" in df.columns and "id_vehicule" in df_usagers.columns:
    join_keys.append("id_vehicule")

df = df.merge(
    df_usagers[usager_cols],
    on=join_keys,
    how="left",
    suffixes=("", "_usr"),
)

print(f"✓ Table consolidée : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"  Accidents uniques : {df['num_acc'].nunique():,}")
print(f"  Période : {df['an'].min()} – {df['an'].max()}")

### 4.1 Valeurs manquantes

In [ ]:
# Taux de valeurs manquantes par colonne
missing = (
    df.isnull().sum()
    .to_frame("nb_missing")
    .assign(pct_missing=lambda x: round(100 * x["nb_missing"] / len(df), 1))
    .query("nb_missing > 0")
    .sort_values("pct_missing", ascending=False)
)

print(f"Colonnes avec valeurs manquantes : {len(missing)} / {df.shape[1]}")
display(missing.head(20))

## 5. Analyse exploratoire

### 5.1 Évolution temporelle

In [ ]:
# Nombre d'accidents et de tués par an
yearly = (
    df.groupby("an")
    .agg(
        nb_accidents=("num_acc", "nunique"),
        nb_tues=("grav", lambda x: (x == 2).sum()),
        nb_blesses=("grav", lambda x: x.isin([3, 4]).sum()),
    )
    .reset_index()
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=yearly["an"], y=yearly["nb_accidents"],
    name="Accidents", marker_color="#636EFA",
))
fig.add_trace(go.Scatter(
    x=yearly["an"], y=yearly["nb_tues"],
    name="Tués", mode="lines+markers",
    marker_color="#EF553B", yaxis="y2",
))
fig.update_layout(
    title="Évolution annuelle des accidents et des décès (2009–2023)",
    xaxis_title="Année",
    yaxis_title="Nombre d'accidents",
    yaxis2=dict(title="Nombre de tués", overlaying="y", side="right"),
    template="plotly_white",
    height=500,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

# Sauvegarder pour le portfolio
fig.write_image(ASSETS / "01_evolution_annuelle.png", scale=2)

### 5.2 Saisonnalité — Distribution mensuelle

In [ ]:
# Accidents par mois (moyenne sur toutes les années)
monthly = (
    df.groupby("mois")
    .agg(
        nb_accidents=("num_acc", "nunique"),
        nb_tues=("grav", lambda x: (x == 2).sum()),
    )
    .reset_index()
)

MOIS_LABELS = {
    1: "Jan", 2: "Fév", 3: "Mar", 4: "Avr", 5: "Mai", 6: "Juin",
    7: "Juil", 8: "Août", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Déc",
}
monthly["mois_label"] = monthly["mois"].map(MOIS_LABELS)

fig = px.bar(
    monthly, x="mois_label", y="nb_accidents",
    title="Distribution mensuelle des accidents (cumul 2009–2023)",
    labels={"mois_label": "Mois", "nb_accidents": "Nombre d'accidents"},
    color="nb_tues", color_continuous_scale="OrRd",
    template="plotly_white",
)
fig.update_layout(
    coloraxis_colorbar_title="Nb tués",
    height=450,
)
fig.show()

### 5.3 Accidents par créneau horaire

In [ ]:
# Distribution horaire
hourly = (
    df.groupby("hrmn_heure")
    .agg(
        nb_accidents=("num_acc", "nunique"),
        nb_tues=("grav", lambda x: (x == 2).sum()),
    )
    .reset_index()
    .dropna(subset=["hrmn_heure"])
)

hourly["taux_letalite"] = round(100 * hourly["nb_tues"] / hourly["nb_accidents"], 2)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=hourly["hrmn_heure"], y=hourly["nb_accidents"],
    name="Accidents", marker_color="#636EFA", opacity=0.7,
))
fig.add_trace(go.Scatter(
    x=hourly["hrmn_heure"], y=hourly["taux_letalite"],
    name="Taux de létalité (%)", mode="lines+markers",
    marker_color="#EF553B", yaxis="y2",
))
fig.update_layout(
    title="Accidents et létalité par créneau horaire",
    xaxis_title="Heure",
    yaxis_title="Nombre d'accidents",
    yaxis2=dict(title="Taux de létalité (%)", overlaying="y", side="right"),
    template="plotly_white",
    height=500,
)
fig.show()

fig.write_image(ASSETS / "02_distribution_horaire.png", scale=2)

### 5.4 Jour de la semaine

In [ ]:
if "jour_semaine" in df.columns:
    # Ordre des jours
    DAYS_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    DAYS_FR = {
        "Monday": "Lundi", "Tuesday": "Mardi", "Wednesday": "Mercredi",
        "Thursday": "Jeudi", "Friday": "Vendredi", "Saturday": "Samedi", "Sunday": "Dimanche",
    }

    daily = (
        df.groupby("jour_semaine")
        .agg(
            nb_accidents=("num_acc", "nunique"),
            nb_tues=("grav", lambda x: (x == 2).sum()),
        )
        .reindex(DAYS_ORDER)
        .reset_index()
    )
    daily["jour_fr"] = daily["jour_semaine"].map(DAYS_FR)
    daily["taux_letalite"] = round(100 * daily["nb_tues"] / daily["nb_accidents"], 2)

    fig = px.bar(
        daily, x="jour_fr", y="nb_accidents",
        title="Accidents par jour de la semaine",
        labels={"jour_fr": "", "nb_accidents": "Nombre d'accidents"},
        color="taux_letalite", color_continuous_scale="OrRd",
        template="plotly_white",
    )
    fig.update_layout(coloraxis_colorbar_title="Létalité (%)", height=450)
    fig.show()

### 5.5 Facteurs de risque — Analyses bivariées

#### Luminosité × Gravité

In [ ]:
LUM_LABELS = {
    1: "Plein jour", 2: "Crépuscule/Aube",
    3: "Nuit sans éclairage", 4: "Nuit éclairage éteint", 5: "Nuit éclairage allumé",
}

if "lum" in df.columns:
    df["lum_label"] = df["lum"].map(LUM_LABELS)

    lum_grav = (
        df.groupby("lum_label")
        .agg(
            nb_accidents=("num_acc", "nunique"),
            nb_tues=("grav", lambda x: (x == 2).sum()),
        )
        .reset_index()
    )
    lum_grav["taux_letalite"] = round(100 * lum_grav["nb_tues"] / lum_grav["nb_accidents"], 2)

    fig = px.bar(
        lum_grav.sort_values("taux_letalite", ascending=True),
        x="taux_letalite", y="lum_label",
        orientation="h",
        title="Taux de létalité par condition de luminosité",
        labels={"taux_letalite": "Taux de létalité (%)", "lum_label": ""},
        color="taux_letalite", color_continuous_scale="OrRd",
        template="plotly_white",
    )
    fig.update_layout(height=400, showlegend=False)
    fig.show()

    fig.write_image(ASSETS / "03_luminosite_letalite.png", scale=2)

#### Conditions météo × Gravité

In [ ]:
ATM_LABELS = {
    1: "Normale", 2: "Pluie légère", 3: "Pluie forte",
    4: "Neige/Grêle", 5: "Brouillard/Fumée", 6: "Vent fort/Tempête",
    7: "Temps éblouissant", 8: "Temps couvert", 9: "Autre",
}

if "atm" in df.columns:
    df["atm_label"] = df["atm"].map(ATM_LABELS)

    atm_grav = (
        df.groupby("atm_label")
        .agg(
            nb_accidents=("num_acc", "nunique"),
            nb_tues=("grav", lambda x: (x == 2).sum()),
        )
        .reset_index()
        .dropna()
    )
    atm_grav["taux_letalite"] = round(100 * atm_grav["nb_tues"] / atm_grav["nb_accidents"], 2)

    fig = px.bar(
        atm_grav.sort_values("taux_letalite", ascending=True),
        x="taux_letalite", y="atm_label",
        orientation="h",
        title="Taux de létalité par conditions atmosphériques",
        labels={"taux_letalite": "Taux de létalité (%)", "atm_label": ""},
        color="nb_accidents", color_continuous_scale="Blues",
        template="plotly_white",
    )
    fig.update_layout(height=450, coloraxis_colorbar_title="Nb accidents")
    fig.show()

#### Catégorie de route × Gravité

In [ ]:
if "route_label" in df.columns:
    route_grav = (
        df.groupby("route_label")
        .agg(
            nb_accidents=("num_acc", "nunique"),
            nb_tues=("grav", lambda x: (x == 2).sum()),
        )
        .reset_index()
        .dropna()
    )
    route_grav["taux_letalite"] = round(100 * route_grav["nb_tues"] / route_grav["nb_accidents"], 2)

    fig = px.bar(
        route_grav.sort_values("taux_letalite", ascending=True),
        x="taux_letalite", y="route_label",
        orientation="h",
        title="Taux de létalité par catégorie de route",
        labels={"taux_letalite": "Taux de létalité (%)", "route_label": ""},
        color="taux_letalite", color_continuous_scale="OrRd",
        template="plotly_white",
    )
    fig.update_layout(height=400)
    fig.show()

### 5.6 Profils des usagers

In [ ]:
# Gravité par tranche d'âge
if "tranche_age" in df.columns:
    age_grav = (
        df.groupby("tranche_age", observed=True)
        .agg(
            nb_usagers=("grav", "count"),
            nb_tues=("grav", lambda x: (x == 2).sum()),
        )
        .reset_index()
    )
    age_grav["taux_letalite"] = round(100 * age_grav["nb_tues"] / age_grav["nb_usagers"], 2)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=age_grav["tranche_age"].astype(str),
        y=age_grav["nb_usagers"],
        name="Usagers impliqués",
        marker_color="#636EFA", opacity=0.7,
    ))
    fig.add_trace(go.Scatter(
        x=age_grav["tranche_age"].astype(str),
        y=age_grav["taux_letalite"],
        name="Taux de létalité (%)",
        mode="lines+markers",
        marker_color="#EF553B", yaxis="y2",
    ))
    fig.update_layout(
        title="Implication et létalité par tranche d'âge",
        xaxis_title="Tranche d'âge",
        yaxis_title="Nombre d'usagers",
        yaxis2=dict(title="Taux de létalité (%)", overlaying="y", side="right"),
        template="plotly_white", height=500,
    )
    fig.show()

    fig.write_image(ASSETS / "04_age_letalite.png", scale=2)

In [ ]:
# Gravité par catégorie d'usager et sexe
if all(col in df.columns for col in ["catu_label", "sexe_label"]):
    cross = (
        df.groupby(["catu_label", "sexe_label"])
        .agg(
            nb_usagers=("grav", "count"),
            nb_tues=("grav", lambda x: (x == 2).sum()),
        )
        .reset_index()
        .dropna()
    )
    cross["taux_letalite"] = round(100 * cross["nb_tues"] / cross["nb_usagers"], 2)

    fig = px.bar(
        cross, x="catu_label", y="taux_letalite",
        color="sexe_label", barmode="group",
        title="Taux de létalité par catégorie d'usager et sexe",
        labels={
            "catu_label": "Catégorie d'usager",
            "taux_letalite": "Taux de létalité (%)",
            "sexe_label": "Sexe",
        },
        color_discrete_map={"Homme": "#636EFA", "Femme": "#EF553B"},
        template="plotly_white",
    )
    fig.update_layout(height=450)
    fig.show()

### 5.7 Analyse géographique

In [ ]:
# Top 15 départements par nombre de tués
if "dep" in df.columns:
    dep_stats = (
        df.groupby("dep")
        .agg(
            nb_accidents=("num_acc", "nunique"),
            nb_tues=("grav", lambda x: (x == 2).sum()),
        )
        .reset_index()
        .sort_values("nb_tues", ascending=False)
        .head(15)
    )

    fig = px.bar(
        dep_stats.sort_values("nb_tues"),
        x="nb_tues", y="dep",
        orientation="h",
        title="Top 15 départements par nombre de tués (2009–2023)",
        labels={"nb_tues": "Nombre de tués", "dep": "Département"},
        color="nb_tues", color_continuous_scale="Reds",
        template="plotly_white",
    )
    fig.update_layout(height=500, showlegend=False)
    fig.show()

    fig.write_image(ASSETS / "05_top_departements.png", scale=2)

In [ ]:
# Carte de chaleur si coordonnées disponibles
if "lat" in df.columns and "long" in df.columns:
    import folium
    from folium.plugins import HeatMap

    # Échantillon pour la carte (trop de points sinon)
    tues = df[df["grav"] == 2].dropna(subset=["lat", "long"])
    
    # Filtrer les coordonnées en France métropolitaine
    tues_metro = tues[
        (tues["lat"].between(41, 52)) &
        (tues["long"].between(-5, 10))
    ]

    # Créer la carte
    m = folium.Map(location=[46.5, 2.5], zoom_start=6, tiles="CartoDB positron")
    
    heat_data = tues_metro[["lat", "long"]].values.tolist()
    HeatMap(heat_data, radius=8, blur=10, max_zoom=10).add_to(m)

    # Sauvegarder
    m.save(ASSETS / "06_carte_chaleur_tues.html")
    print(f"✓ Carte de chaleur sauvegardée ({len(heat_data):,} points)")
    print(f"  → Ouvrir {ASSETS / '06_carte_chaleur_tues.html'} dans un navigateur")
    
    m  # Affichage dans le notebook

## 6. Requêtage SQL

Export vers SQLite et exécution des requêtes KPIs définies dans `sql/queries_kpis.sql`.

In [ ]:
# Créer la base SQLite en mémoire
conn = sqlite3.connect(":memory:")

# Colonnes à exporter (les plus utiles)
EXPORT_COLS = [
    "num_acc", "an", "mois", "jour", "hrmn_heure", "lum", "dep", "agg", "col", "atm",
    "catr", "surf",
    "grav", "sexe", "age", "catu", "tranche_age",
    "lat", "long",
]

# Ne garder que les colonnes qui existent
export_cols_existing = [c for c in EXPORT_COLS if c in df.columns]
df_export = df[export_cols_existing].copy()

# Écrire dans SQLite
df_export.to_sql("accidents", conn, index=False, if_exists="replace")
print(f"✓ Table 'accidents' créée dans SQLite : {len(df_export):,} lignes")

In [ ]:
# Exécuter quelques requêtes clés

# KPI 1 : Évolution annuelle
query_evolution = """
SELECT
    an AS annee,
    COUNT(DISTINCT Num_Acc) AS nb_accidents,
    SUM(CASE WHEN grav = 2 THEN 1 ELSE 0 END) AS nb_tues,
    ROUND(
        100.0 * SUM(CASE WHEN grav = 2 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0), 2
    ) AS taux_letalite_pct
FROM accidents
GROUP BY an
ORDER BY an
"""

print("📊 Évolution annuelle")
display(pd.read_sql(query_evolution, conn))

In [ ]:
# KPI 2 : Facteurs de risque par luminosité
query_lum = """
SELECT
    CASE lum
        WHEN 1 THEN 'Plein jour'
        WHEN 2 THEN 'Crépuscule / Aube'
        WHEN 3 THEN 'Nuit sans éclairage'
        WHEN 4 THEN 'Nuit éclairage éteint'
        WHEN 5 THEN 'Nuit éclairage allumé'
        ELSE 'Inconnu'
    END AS luminosite,
    COUNT(DISTINCT Num_Acc) AS nb_accidents,
    SUM(CASE WHEN grav = 2 THEN 1 ELSE 0 END) AS nb_tues,
    ROUND(
        100.0 * SUM(CASE WHEN grav = 2 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0), 2
    ) AS taux_letalite_pct
FROM accidents
GROUP BY luminosite
ORDER BY taux_letalite_pct DESC
"""

print("📊 Létalité par luminosité")
display(pd.read_sql(query_lum, conn))

In [ ]:
# KPI 3 : Profil des victimes tuées
query_profil = """
SELECT
    CASE catu
        WHEN 1 THEN 'Conducteur'
        WHEN 2 THEN 'Passager'
        WHEN 3 THEN 'Piéton'
        ELSE 'Autre'
    END AS categorie,
    COUNT(*) AS nb_tues,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM accidents WHERE grav = 2), 1) AS pct_total
FROM accidents
WHERE grav = 2
GROUP BY categorie
ORDER BY nb_tues DESC
"""

print("📊 Répartition des tués par catégorie d'usager")
display(pd.read_sql(query_profil, conn))

conn.close()

## 7. Export pour Looker Studio

Google Sheets a une limite de ~5 millions de cellules. Le fichier au grain usager dépasse cette limite.

**Stratégie :** on crée **deux exports** :
1. **`accidents_detail.csv`** — grain *accident* (1 ligne = 1 accident), avec les compteurs agrégés (nb tués, blessés, etc.) + pire gravité. Pour Google Sheets → Looker Studio.
2. **`accidents_usagers_full.parquet`** — grain *usager* complet, pour analyses locales uniquement.

L'export accident-level couvre 15 ans et donne ~1M lignes × ~15 colonnes ≈ 15M cellules.
Si c'est encore trop pour Google Sheets, on filtre sur les 5 dernières années (~350k lignes).


In [ ]:
# ============================================================
# EXPORT 1 : Grain ACCIDENT (pour Google Sheets → Looker Studio)
# ============================================================

# Agréger au niveau accident : 1 ligne = 1 accident
# Avec compteurs de victimes et pire gravité

accident_agg = (
    df.groupby("num_acc")
    .agg(
        # Temporel
        Annee=("an", "first"),
        Mois=("mois", "first"),
        Jour=("jour", "first"),
        Heure=("hrmn_heure", "first"),
        Jour_Semaine=("jour_semaine", "first") if "jour_semaine" in df.columns else ("an", "first"),
        # Géographie
        Departement=("dep", "first"),
        # Conditions
        Luminosite=("lum_label", "first") if "lum_label" in df.columns else ("lum", "first"),
        Meteo=("atm_label", "first") if "atm_label" in df.columns else ("atm", "first"),
        Agglomeration=("agg", "first"),
        # Infrastructure
        Type_Route=("route_label", "first") if "route_label" in df.columns else ("catr", "first"),
        Etat_Surface=("surface_label", "first") if "surface_label" in df.columns else ("surf", "first"),
        # Collision
        Type_Collision=("col", "first"),
        # Géolocalisation
        Latitude=("lat", "first"),
        Longitude=("long", "first"),
        # Compteurs usagers
        Nb_Usagers=("grav", "count"),
        Nb_Tues=("grav", lambda x: (x == 2).sum()),
        Nb_Hospitalises=("grav", lambda x: (x == 3).sum()),
        Nb_Blesses_Legers=("grav", lambda x: (x == 4).sum()),
        Nb_Indemnes=("grav", lambda x: (x == 1).sum()),
        # Pire gravité de l'accident
        Gravite_Max=("grav", "min"),  # 2=Tué est le min numérique = le pire
    )
    .reset_index()
)

# Labels utiles
GRAV_MAX_LABELS = {1: "Indemne", 2: "Mortel", 3: "Hospitalisé", 4: "Blessé léger"}
accident_agg["Gravite_Label"] = accident_agg["Gravite_Max"].map(GRAV_MAX_LABELS)

COL_LABELS = {
    1: "Frontale", 2: "Par l arrière", 3: "Par le côté",
    4: "En chaîne", 5: "Collisions multiples", 6: "Autre", 7: "Sans collision",
}
accident_agg["Collision_Label"] = accident_agg["Type_Collision"].map(COL_LABELS)

AGG_LABELS = {1: "Hors agglomération", 2: "En agglomération"}
accident_agg["Milieu"] = accident_agg["Agglomeration"].map(AGG_LABELS)

# Flags binaires pour les métriques Looker Studio
accident_agg["Est_Mortel"] = (accident_agg["Gravite_Max"] == 2).astype(int)

print(f"✓ Export grain accident : {len(accident_agg):,} lignes × {accident_agg.shape[1]} colonnes")
print(f"  Cellules : {len(accident_agg) * accident_agg.shape[1]:,}")

# Vérifier si ça passe dans Google Sheets (limite ~5M cellules)
nb_cellules = len(accident_agg) * accident_agg.shape[1]
if nb_cellules > 5_000_000:
    print(f"\n⚠️  Trop de cellules ({nb_cellules:,}) pour Google Sheets.")
    print("  → Filtrage sur les 5 dernières années...")
    annee_max = accident_agg["Annee"].max()
    accident_agg = accident_agg[accident_agg["Annee"] >= annee_max - 4]
    print(f"  → {len(accident_agg):,} lignes ({accident_agg['Annee'].min()}-{accident_agg['Annee'].max()})")
    print(f"  → Cellules : {len(accident_agg) * accident_agg.shape[1]:,}")
else:
    print(f"  ✓ OK pour Google Sheets")

# Sauvegarder le CSV pour Looker Studio
output_csv = DATA_PROCESSED / "accidents_detail.csv"
accident_agg.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"\n✓ Fichier CSV exporté : {output_csv}")
print(f"  Taille : {output_csv.stat().st_size / 1e6:.1f} Mo")

# ============================================================
# EXPORT 2 : Grain USAGER complet (pour analyse locale)
# ============================================================

# output_parquet = DATA_PROCESSED / "accidents_usagers_full.parquet"
# df.to_parquet(output_parquet, index=False)
# print(f"\n✓ Fichier Parquet complet : {output_parquet}")
# print(f"  {len(df):,} lignes (grain usager)")

display(accident_agg.head())


## 8. Import dans Looker Studio

Le fichier `data/processed/accidents_detail.csv` est prêt.

### Étapes :
1. **Google Sheets** : Importer le CSV (Fichier → Importer → Upload)
2. **Looker Studio** : Créer un rapport → Ajouter des données → Google Sheets
3. Suivre le guide `GUIDE_LOOKER_STUDIO.md` pour construire les 4 pages

### Colonnes disponibles :

| Colonne | Type | Usage |
|---------|------|-------|
| Annee, Mois, Jour, Heure | Dimensions temporelles | Filtres + axes |
| Departement | Dimension géo | Carte + filtres |
| Luminosite, Meteo, Type_Route, Etat_Surface | Facteurs de risque | Analyses bivariées |
| Nb_Tues, Nb_Hospitalises, Nb_Blesses_Legers | Métriques | Compteurs |
| Est_Mortel | Flag binaire | SUM pour compter les accidents mortels |
| Gravite_Label | Dimension | Filtres + répartition |
| Latitude, Longitude | Géo | Carte Google Maps |


---

*Notebook réalisé par Diogo Almeida — Septembre 2026*  
*Données : Ministère de l'Intérieur / ONISR — Licence Ouverte*